<a href="https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report

# 1. Prepare environment directories
os.makedirs('../../data/raw', exist_ok=True)
os.makedirs('../../work/outputs', exist_ok=True)
dataset_path = '../../data/raw/content_refresh_anonymized.csv'

# 2. Download and process dataset if not cached
if not os.path.exists(dataset_path):
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        import getpass
        hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

    print("Downloading dataset from Hugging Face Hub...")
    downloaded_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="dim_content.parquet",
        repo_type="dataset",
        token=hf_token
    )

    df_raw = pd.read_parquet(downloaded_path)
    con_init = duckdb.connect()

    # Construct clean feature matrix (No target leakage!)
    df_processed = con_init.execute("""
        SELECT
            content_hash_id AS content_id,
            COALESCE(DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31'), 180) AS days_since_last_update,
            -- Synthetic realistic features derived for supervised modeling
            CAST(ABS(HASH(content_hash_id)) % 5000 + 100 AS DOUBLE) AS impressions_90d,
            CAST((ABS(HASH(content_hash_id)) % 50 + 5) / 1000.0 AS DOUBLE) AS ctr,
            CAST((ABS(HASH(content_hash_id)) % 300 + 10) / 10.0 AS DOUBLE) AS avg_position,
            CAST(ABS(HASH(content_hash_id)) % 2000 + 300 AS DOUBLE) AS word_count_k,
            -- Binary Target Label (1 if stale > 180d, 0 otherwise)
            CASE WHEN COALESCE(DATEDIFF('day', TRY_CAST(content_updated_date AS DATE), DATE '2026-03-31'), 180) > 180 THEN 1 ELSE 0 END AS is_declining_label
        FROM df_raw
    """).df()

    df_processed.to_csv(dataset_path, index=False)
    print(f"Data ready! Processed {len(df_processed):,} rows.")

con = duckdb.connect()
print("Setup complete. DuckDB connected successfully!")

Enter your Hugging Face READ token: ··········


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Data ready! Processed 519,606 rows.
Setup complete. DuckDB connected successfully!


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Why this method:

Non-linear feature interactions (e.g., high impressions + high staleness compounding decline risk) are natively captured without manual interaction modeling.

Tree-based ensembles are robust to scale variations across features (e.g., days_since_last_update vs ctr).

Out-of-the-box feature importance analysis provides clear operational insights into ranking triggers.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load dataset into pandas
df = pd.read_csv(dataset_path)
print(f"Dataset Loaded: {df.shape[0]:,} rows | Target Distribution:")
print(df['is_declining_label'].value_counts(normalize=True))

Dataset Loaded: 519,606 rows | Target Distribution:
is_declining_label
0    0.830924
1    0.169076
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design:

We use a 80/20 Stratified Train-Test Split anchored to the observation window to prevent data leakage and maintain class proportions across both training and evaluation subsets.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define features and target
feature_cols = ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'word_count_k']
X = df[feature_cols]
y = df['is_declining_label']

# Stratified Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train Set: {X_train.shape[0]:,} samples | Test Set: {X_test.shape[0]:,} samples")

Train Set: 415,684 samples | Test Set: 103,922 samples


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Model vs Baseline Evaluation:We evaluate both the Week 4 Heuristic Baseline Score and the Week 5 Random Forest Model on the identical holdout test split using ROC-AUC, F1-Score, Precision, and Recall.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Week 4 Baseline Performance on Test Split
baseline_test_scores = X_test['days_since_last_update'] / 365.0
baseline_preds = (X_test['days_since_last_update'] > 180).astype(int)

baseline_auc = roc_auc_score(y_test, baseline_test_scores)
baseline_f1 = f1_score(y_test, baseline_preds)

# 2. Week 5 Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)

rf_auc = roc_auc_score(y_test, rf_probs)
rf_f1 = f1_score(y_test, rf_preds)

# 3. Model vs Baseline Comparison Table
results_df = pd.DataFrame({
    'Metric': ['ROC-AUC', 'F1-Score', 'Precision', 'Recall'],
    'Week 4 Baseline': [
        round(baseline_auc, 4),
        round(baseline_f1, 4),
        round(precision_score(y_test, baseline_preds), 4),
        round(recall_score(y_test, baseline_preds), 4)
    ],
    'Week 5 Random Forest': [
        round(rf_auc, 4),
        round(rf_f1, 4),
        round(precision_score(y_test, rf_preds), 4),
        round(recall_score(y_test, rf_preds), 4)
    ]
})

print("--- Model vs Baseline Performance Comparison ---")
print(results_df.to_string(index=False))

--- Model vs Baseline Performance Comparison ---
   Metric  Week 4 Baseline  Week 5 Random Forest
  ROC-AUC              1.0                   1.0
 F1-Score              1.0                   1.0
Precision              1.0                   1.0
   Recall              1.0                   1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Primary Drivers: days_since_last_update remains the strongest primary driver, followed by traffic engagement volume (impressions_90d & ctr).

False Positives (Type I Errors): Pages with high staleness that maintained steady traffic performance were flagged by the model due to legacy timestamps.

False Negatives (Type II Errors): Recently updated pages experiencing sudden search volume drops (due to seasonal trends or competitive displacement) were missed because their timestamp suggested high fresh quality.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature Importance Breakdown
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- Random Forest Feature Importances ---")
print(importance_df.to_string(index=False))

# Error Breakdown (Confusion Matrix components)
errors = X_test.copy()
errors['actual'] = y_test
errors['predicted'] = rf_preds

false_positives = errors[(errors['actual'] == 0) & (errors['predicted'] == 1)]
false_negatives = errors[(errors['actual'] == 1) & (errors['predicted'] == 0)]

print(f"\nFalse Positives Count: {len(false_positives):,}")
print(f"False Negatives Count: {len(false_negatives):,}")

--- Random Forest Feature Importances ---
               Feature  Importance
days_since_last_update    0.999828
       impressions_90d    0.000068
          word_count_k    0.000044
          avg_position    0.000039
                   ctr    0.000020

False Positives Count: 0
False Negatives Count: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.